In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pickle

np.random.seed(42)


# === PARAMETRY — zmień tutaj ===
N_NORMAL = 2000      # liczba normalnych transakcji
N_FRAUD  = 100       # liczba fraudów
# ===============================

# Normalne transakcje
normal = pd.DataFrame({
    'amount': np.random.lognormal(5, 1, N_NORMAL).clip(5, 5000),
    'is_electronics': np.random.binomial(1, 0.3, N_NORMAL),
    'tx_per_minute': np.random.poisson(3, N_NORMAL),
    'fraud': 0
})


# Fraudy
fraud = pd.DataFrame({
    'amount': np.random.uniform(2000, 9000, N_FRAUD),
    'is_electronics': np.random.binomial(1, 0.7, N_FRAUD),
    'tx_per_minute': np.random.poisson(8, N_FRAUD),
    'fraud': 1
})

df = pd.concat([normal, fraud], ignore_index=True).sample(frac=1, random_state=42)
print(f"Dataset: {len(df)} wierszy, fraud rate: {df['fraud'].mean():.1%}")

# Trening modelu
features = ['amount', 'is_electronics', 'tx_per_minute']
X, y = df[features], df['fraud']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)

print(classification_report(y_test, model.predict(X_test)))

# Zapisanie modelu do pliku
with open('fraud_model.pkl', 'wb') as f:
    pickle.dump(model, f)

Dataset: 2100 wierszy, fraud rate: 4.8%
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       400
           1       1.00      1.00      1.00        20

    accuracy                           1.00       420
   macro avg       1.00      1.00      1.00       420
weighted avg       1.00      1.00      1.00       420



In [2]:
%%file fraud_api.py
from fastapi import FastAPI
from pydantic import BaseModel
import pickle
import numpy as np

app = FastAPI(title="Fraud Detection API")

with open('fraud_model.pkl', 'rb') as f:
    model = pickle.load(f)

class Transaction(BaseModel):
    amount: float
    is_electronics: int
    tx_per_minute: int

@app.get("/health")
def health():
    return {"status": "ok", "model_loaded": True}

@app.post("/score")
def score(tx: Transaction):
    features = np.array([[tx.amount, tx.is_electronics, tx.tx_per_minute]])
    
    prediction = model.predict(features)[0]
    probability = model.predict_proba(features)[0][1]
    
    return {
        "is_fraud": bool(prediction),
        "fraud_probability": float(probability)
    }

Writing fraud_api.py


In [3]:
import requests

# Test normalna
r = requests.post("http://localhost:8001/score",
    json={"amount": 150, "is_electronics": 0, "tx_per_minute": 3})
print("Normalna:", r.json())

# 2. Test podejrzanej transakcji
r_fraud = requests.post("http://localhost:8001/score",
    json={"amount": 5500, "is_electronics": 1, "tx_per_minute": 12})
print("Podejrzana:", r_fraud.json())

print(r_fraud.json())

Normalna: {'is_fraud': False, 'fraud_probability': 0.0}
Podejrzana: {'is_fraud': True, 'fraud_probability': 1.0}
{'is_fraud': True, 'fraud_probability': 1.0}


In [4]:
%%file ml_consumer.py
from kafka import KafkaConsumer, KafkaProducer
from datetime import datetime
import json, requests

consumer = KafkaConsumer('transactions', bootstrap_servers='broker:9092',
    auto_offset_reset='earliest', group_id='ml-scoring',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')))

alert_producer = KafkaProducer(bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8'))

API_URL = "http://localhost:8001/score"

print("Konsument ML wystartował i nasłuchuje na temat: transactions...")

for message in consumer:
    tx = message.value
    
    features = {
        "amount": tx.get('amount'),
        "is_electronics": tx.get('is_electronics'),
        "tx_per_minute": tx.get('tx_per_minute', 5)
    }
    
    try:
        r = requests.post(API_URL, json=features)
        result = r.json()
        
        if result.get("is_fraud"):
            print(f"!!! ALERT !!! Wykryto fraud: {tx['amount']} PLN | Prawdopodobieństwo: {result['fraud_probability']}")
            
            alert_payload = {
                "timestamp": datetime.now().isoformat(),
                "status": "FRAUD_ALERT",
                "details": result,
                "transaction_data": tx
            }
            alert_producer.send('alerts', alert_payload)
            
    except Exception as e:
        print(f"Błąd podczas scoringu: {e}")

Writing ml_consumer.py


In [5]:
import requests
r = requests.get("http://localhost:8001/openapi.json")
print(r.json()['components']['schemas']['Transaction']['properties'])

{'amount': {'type': 'number', 'title': 'Amount'}, 'is_electronics': {'type': 'integer', 'title': 'Is Electronics'}, 'tx_per_minute': {'type': 'integer', 'title': 'Tx Per Minute'}}


In [6]:
%%file ml_consumer.py
from kafka import KafkaConsumer, KafkaProducer
import json, requests

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='latest', 
    group_id='ml-scoring-final',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

API_URL = "http://localhost:8001/score"

print("Konsument ML startuje...")

for message in consumer:
    tx = message.value
    
    is_elec = 1 if tx.get('category') == 'elektronika' else 0
    
    payload = {
        "amount": float(tx.get('amount', 0)),
        "is_electronics": int(is_elec),
        "tx_per_minute": int(tx.get('tx_per_minute', 5))
    }
    
    try:
        r = requests.post(API_URL, json=payload)
        if r.status_code == 200:
            res = r.json()
            status = "FRAUD" if res['is_fraud'] else "OK"
            print(f"Kwota: {payload['amount']} PLN | Kat: {tx.get('category')} | Wynik: {status}")
        else:
            print(f"BŁĄD API {r.status_code}: {r.text}")
    except Exception as e:
        print(f"Błąd: {e}")

Overwriting ml_consumer.py


In [7]:
%%file fraud_api.py
from fastapi import FastAPI
from pydantic import BaseModel
import pickle
import numpy as np

app = FastAPI()

# Ładowanie modelu
with open('fraud_model.pkl', 'rb') as f:
    model = pickle.load(f)

class Transaction(BaseModel):
    amount: float
    is_electronics: int
    tx_per_minute: int

@app.get("/health")
def health():
    """Endpoint do sprawdzania czy API żyje"""
    return {
        "status": "healthy",
        "model_loaded": True,
        "service": "fraud-detection-ml"
    }

@app.post("/score")
def score(tx: Transaction):
    features = np.array([[tx.amount, tx.is_electronics, tx.tx_per_minute]])
    prediction = model.predict(features)[0]
    probability = model.predict_proba(features)[0][1]
    return {
        "is_fraud": bool(prediction), 
        "fraud_probability": float(probability)
    }

Overwriting fraud_api.py
